# Models Training

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.neural_network import MLPClassifier
import plotly.io as pio

sns.set_style(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [4]:
pio.renderers.default = "vscode"
pd.set_option('display.max_columns', None)

In [5]:
try:
    df = pd.read_csv('./preprocessed_cad.csv')
    df.columns = [c.lower() for c in df.columns]
    print(f"✅ Dataset Loaded: {df.shape[0]} patients, {df.shape[1]} columns.")
except FileNotFoundError:
    print("❌ Error. preprocessed_cad file not found")

✅ Dataset Loaded: 432 patients, 37 columns.


In [7]:
X = df.drop('cath', axis=1)
y = df['cath']

## K-Fold Cross Validation

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

model_cv = LogisticRegression(max_iter=1000, solver='liblinear')

cv_results = cross_val_score(model_cv, X_train, y_train, cv=kf, scoring='accuracy')

fig = go.Figure()
fig.add_trace(go.Box(y=cv_results, name='Logistic Regression CV', boxpoints='all', jitter=0.3, pointpos=-1.8))
fig.update_layout(title='Real Distribution (K-Fold)', yaxis_title='Accuracy')
fig.show()

print(f"✅ Mean Accuracy: {cv_results.mean():.4f}")
print(f"📉 Standard Deviation: {cv_results.std():.4f}")

✅ Mean Accuracy: 0.8960
📉 Standard Deviation: 0.0423


In [12]:
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(X,
                                                                            y,
                                                                            test_size=0.2,
                                                                            shuffle=True,
                                                                            stratify=y,
                                                                            random_state=42)

## Poly-LogReg Pipeline

In [9]:
class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bound_ = {}
        self.upper_bound_ = {}

    def fit(self, X, y=None):
        
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        for col in X.columns:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            self.lower_bound_[col] = Q1 - (self.factor * IQR)
            self.upper_bound_[col] = Q3 + (self.factor * IQR)
        return self

    def transform(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        X_capped = X.copy()
        for col in X.columns:
            if col in self.lower_bound_:
                X_capped[col] = np.clip(X_capped[col], self.lower_bound_[col], self.upper_bound_[col])
        return X_capped.values 

print("✅'OutlierCapper' defined!")

✅'OutlierCapper' defined!


In [19]:
poly_pipeline = Pipeline([

    # 1: Outlier Capping 
    ('outlier_capper', OutlierCapper(factor=1.5)),

    # 2: Polynomials
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),

    # 3: Scaling
    ('scaler', StandardScaler()),

    # 4: Regularization
    ('classifier', LogisticRegression(solver='liblinear', max_iter=3000))
])

print("✅ Pipeline:  Cap -> Poly -> Scale -> LogReg")

✅ Pipeline:  Cap -> Poly -> Scale -> LogReg


In [22]:
param_grid_poly = {

    'poly__degree': [1, 2, 3],

    'classifier__C': [0.01, 0.1, 1, 10, 100],

    'classifier__penalty': ['l1', 'l2']
}

cv = StratifiedKFold(n_splits=5,
           shuffle=True,
           random_state=42)

grid_poly = GridSearchCV(poly_pipeline,
                         param_grid_poly,
                         cv=cv,
                         scoring='accuracy',
                         n_jobs=-1,
                         verbose=1)

grid_poly.fit(X_train_final, y_train_final)

print(f"\n🏆 CV best score: {grid_poly.best_score_:.4f}")
print("⚙️ Bst parameters:")
print(grid_poly.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits

🏆 CV best score: 0.9101
⚙️ Bst parameters:
{'classifier__C': 0.1, 'classifier__penalty': 'l2', 'poly__degree': 1}


d:\Programming\artificial_intelligence\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [23]:
print(f"Mean Accuracy (CV): {grid_poly.best_score_:.4f}")
print(f"Stability (Std Dev): {grid_poly.cv_results_['std_test_score'][grid_poly.best_index_]:.4f}")

Mean Accuracy (CV): 0.9101
Stability (Std Dev): 0.0192


In [24]:
best_model = grid_poly.best_estimator_
y_pred_poly = best_model.predict(X_test_final)

In [25]:
# Confusion Matrix for LogReg
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_final, y_pred_poly)

fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                   labels=dict(x="Prediction", y="Reality", color="Number of Patients"),
                   x=['Healthy', 'CAD'], y=['Healthy', 'CAD'],
                   title="Confusion Matrix : Poly-LogReg")
fig_cm.show()

In [43]:
y_proba = best_model.predict_proba(X_test_final)[:, 1] 

threshold = 0.2
y_pred_custom = (y_proba >= threshold).astype(int)

In [45]:

cm = confusion_matrix(y_test_final, y_pred_custom)

fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                   labels=dict(x="Prediction", y="Reality", color="Number of Patients"),
                   x=['Healthy', 'CAD'], y=['Healthy', 'CAD'],
                   title="Confusion Matrix : Poly-LogReg Custom Threshold")
fig_cm.show()

## Decision Tree

In [46]:
pipe_dt = Pipeline([
    ('classifier', DecisionTreeClassifier(random_state=42))
])

grid_dt = GridSearchCV(
    pipe_dt,
    param_grid={
        'classifier__max_depth': [3, 5, 7, 10, None],
        'classifier__min_samples_split': [2, 5, 10, 15],
        'classifier__min_samples_leaf': [1, 5, 7, 10]
    },
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

## Random Forest

In [47]:
pipe_rf = Pipeline([
    ('classifier', RandomForestClassifier(random_state=42))
])

grid_rf = GridSearchCV(
    pipe_rf,
    param_grid={
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [5, 10],
        'classifier__min_samples_leaf': [1, 4]
    },
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

In [48]:
print("⏳ Decision Tree Training...")
grid_dt.fit(X_train_final, y_train_final)

print("⏳ Random Forest Training...")
grid_rf.fit(X_train_final, y_train_final)

⏳ Decision Tree Training...
⏳ Random Forest Training...


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__max_depth': [5, 10], 'classifier__min_samples_leaf': [1, 4], 'classifier__n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation ti

## Model Comparison

In [50]:
print(f"[Decision Tree] Mean Accuracy (CV): {grid_poly.best_score_:.4f}")
print(f"[Decision Tree] Stability (Std Dev): {grid_poly.cv_results_['std_test_score'][grid_poly.best_index_]:.4f}")

[Decision Tree] Mean Accuracy (CV): 0.9101
[Decision Tree] Stability (Std Dev): 0.0192


In [51]:
print(f"[Decision Tree] Mean Accuracy (CV): {grid_dt.best_score_:.4f}")
print(f"[Decision Tree] Stability (Std Dev): {grid_dt.cv_results_['std_test_score'][grid_dt.best_index_]:.4f}")

[Decision Tree] Mean Accuracy (CV): 0.8841
[Decision Tree] Stability (Std Dev): 0.0458


In [52]:
print(f"[Random Forest] Mean Accuracy (CV): {grid_rf.best_score_:.4f}")
print(f"[Random Forest] Stability  (Std Dev): {grid_rf.cv_results_['std_test_score'][grid_rf.best_index_]:.4f}")

[Random Forest] Mean Accuracy (CV): 0.9333
[Random Forest] Stability  (Std Dev): 0.0148


## Final Evaluation

In [53]:
results_final = []
models = {
    'Poly-LogReg': grid_poly,
    'Decision Tree': grid_dt,
    'Random Forest': grid_rf,
}

print("\n📊 Final Results on Test Set:")
print("="*60)

for name, grid_model in models.items():
    # 1. Best CV model
    best_model = grid_model.best_estimator_

    # 2. Test Set Prediction
    y_pred = best_model.predict(X_test_final)

    # 3. Metrix
    acc = accuracy_score(y_test_final, y_pred)
    best_cv_score = grid_model.best_score_ # Scorul mediu din CV

    results_final.append({
        'Model': name,
        'CV Accuracy (Training)': best_cv_score,
        'Test Accuracy (Real)': acc,
        'Best Params': str(grid_model.best_params_)
    })

    print(f"\n🔹 {name}")
    print(f"   Best CV Score: {best_cv_score:.4f} | Test Score: {acc:.4f}")
    print(f"   Params: {grid_model.best_params_}")


📊 Final Results on Test Set:

🔹 Poly-LogReg
   Best CV Score: 0.9101 | Test Score: 0.8966
   Params: {'classifier__C': 0.1, 'classifier__penalty': 'l2', 'poly__degree': 1}

🔹 Decision Tree
   Best CV Score: 0.8841 | Test Score: 0.8046
   Params: {'classifier__max_depth': 7, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 15}

🔹 Random Forest
   Best CV Score: 0.9333 | Test Score: 0.8621
   Params: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 200}


In [56]:
df_results = pd.DataFrame(results_final)

df_melted = df_results.melt(id_vars="Model",
                            value_vars=["CV Accuracy (Training)", "Test Accuracy (Real)"],
                            var_name="Tip Scor", value_name="Acuratețe")

fig = px.bar(df_melted, x="Model", y="Acuratețe", color="Tip Scor", barmode="group",
             text_auto='.3f', title="🏆 Final ranking: CV vs Test Set",
             color_discrete_map={"CV Accuracy (Training)": "lightgray", "Test Accuracy (Real)": "royalblue"},
             range_y=[0.6, 1])

fig.show()


winner = df_results.sort_values(by="Test Accuracy (Real)", ascending=False).iloc[0]
print(f"\n🏆 Winner: {winner['Model']} with a accuracy of {winner['Test Accuracy (Real)']:.4f}")


🏆 Winner: Poly-LogReg with a accuracy of 0.8966


## Stability Testing

In [62]:
from sklearn.base import clone
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split


seeds = [10, 25, 42, 55, 70, 99, 123, 200, 314, 999]
results_stability = []

try:
    model_dt_optim = clone(grid_dt.best_estimator_) 
    model_rf_optim = clone(grid_rf.best_estimator_)
    model_lr_optim = clone(grid_poly.best_estimator_)
    print("✅ Models Cloned!")
except NameError:
    print("❌ EROARE: model not found")

for seed in seeds:
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X, y, test_size=0.2, shuffle=True, random_state=seed, stratify=y
    )

    model_dt_optim.fit(X_train_s, y_train_s)
    model_rf_optim.fit(X_train_s, y_train_s)
    model_lr_optim.fit(X_train_s, y_train_s)

    results_stability.append({
        'Seed': seed,
        'Decision Tree (Tuned)': model_dt_optim.score(X_test_s, y_test_s),
        'Random Forest (Tuned)': model_rf_optim.score(X_test_s, y_test_s),
        'Poly-LogReg (Tuned)': model_lr_optim.score(X_test_s, y_test_s),
    })


df_stab = pd.DataFrame(results_stability)

print("\n📊 Results(Green = the best on that seed):")
display(df_stab.style.highlight_max(axis=1, color='lightgreen', subset=['Decision Tree (Tuned)', 'Random Forest (Tuned)', 'Poly-LogReg (Tuned)']).format("{:.4f}"))

summary = df_stab[['Decision Tree (Tuned)', 'Random Forest (Tuned)', 'Poly-LogReg (Tuned)']].agg(['mean', 'std'])
print("\n📉 Final Statistic:")
display(summary)

fig = px.line(df_stab, x=df_stab.index,
              y=['Decision Tree (Tuned)', 'Random Forest (Tuned)', 'Poly-LogReg (Tuned)'],
              title='🎢 Stability test',
              labels={'value': 'Test Accuracy', 'index': 'Experiment #'},
              markers=True)
fig.show()


fig_box = px.box(df_stab, y=['Decision Tree (Tuned)', 'Random Forest (Tuned)', 'Poly-LogReg (Tuned)'],
                 title='📦 Performance distribution',
                 points="all")
fig_box.show()

✅ Models Cloned!


d:\Programming\artificial_intelligence\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Programming\artificial_intelligence\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Programming\artificial_intelligence\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will


📊 Results(Green = the best on that seed):


d:\Programming\artificial_intelligence\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,Seed,Decision Tree (Tuned),Random Forest (Tuned),Poly-LogReg (Tuned)
0,10.0000,0.8621,0.9310,0.9080
1,25.0000,0.9195,0.9540,0.9540
2,42.0000,0.8046,0.8621,0.8966
3,55.0000,0.8736,0.8966,0.9080
4,70.0000,0.8736,0.8851,0.8736
5,99.0000,0.9080,0.9195,0.8966
6,123.0000,0.8161,0.8966,0.8851
7,200.0000,0.9195,0.9540,0.9310
8,314.0000,0.8506,0.8736,0.8506
9,999.0000,0.8276,0.9080,0.8851



📉 Final Statistic:


,Decision Tree (Tuned),Random Forest (Tuned),Poly-LogReg (Tuned)
mean,0.865517,0.908046,0.898851
std,0.041637,0.031595,0.029078


## Saving models

In [63]:
import pickle

pickle.dump(grid_poly.best_estimator_, open("./models/poly_log_reg.pkl", "wb"))
pickle.dump(grid_rf.best_estimator_, open("./models/random_forest.pkl", "wb"))
pickle.dump(grid_dt.best_estimator_, open("./models/decision_tree.pkl", "wb"))